# Processing Predicates

This notebook demonstrates how to load positive and negative predicates from pickle files, build a domain, and initialize a `LogicModel`.

In [1]:
import pickle
import LogicModel as m
import numpy as np

## 1. Load Data

We load the first 100 items from both the positive (True) and negative (False) predicate lists.

In [2]:
# Load data
with open('Predicates/FILTERED-predicates.pickle', 'rb') as f:
    pos_predicates = pickle.load(f)[:100]

with open('Predicates/FILTERED-negative_predicates.pickle', 'rb') as f:
    neg_predicates = pickle.load(f)[:100]

print(f"Loaded {len(pos_predicates)} positive predicates")
print(f"Loaded {len(neg_predicates)} negative predicates")

Loaded 100 positive predicates
Loaded 100 negative predicates


## 2. Process Data

We need to extract:
1.  **The Domain:** A unique list of all elements (Subjects and Objects) found in both lists.
2.  **Unary Predicates:** A dictionary mapping predicate names to elements. Positive ones get probability `1.0`, negative ones get `0.0`.
3.  **Binary Predicates:** A dictionary mapping predicate names to lists of `(subject, object)` tuples. We only add the positive ones (as absence implies False).

In [3]:
domain_set = set()
unary_preds_dict = {}
binary_preds_dict = {}

def add_to_domain(elem):
    if elem is not None:
        domain_set.add(elem)

# Process Positive Predicates (True)
for subj, pred, obj in pos_predicates:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as simple element (implies prob=1.0)
        unary_preds_dict[pred].append(subj)
    else:
        # Binary
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        binary_preds_dict[pred].append((subj, obj))

# Process Negative Predicates (False)
for subj, pred, obj in neg_predicates:
    add_to_domain(subj)
    add_to_domain(obj)
    
    if obj is None:
        # Unary
        if pred not in unary_preds_dict:
            unary_preds_dict[pred] = []
        # Add as tuple with prob=0.0
        unary_preds_dict[pred].append((subj, 0.0))
    else:
        # Binary
        # Important: Ensure the predicate exists in the dictionary even if the list is empty.
        if pred not in binary_preds_dict:
            binary_preds_dict[pred] = []
        # We do NOT add the pair to the list, so it defaults to False.

domain_list = sorted(list(domain_set))
print(f"Domain Size: {len(domain_list)}")
print(f"Number of Unary Predicates: {len(unary_preds_dict)}")
print(f"Number of Binary Predicates: {len(binary_preds_dict)}")

Domain Size: 159
Number of Unary Predicates: 106
Number of Binary Predicates: 36


## 3. Build Logic Model

In [4]:
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict
)

model.buildAll()

## 4. Verification

We define a helper function `check_truth` to print the result clearly.

In [5]:
def check_unary(pred, elem):
    res = model.unaryOp(pred, elem)
    # res is [[prob_true], [prob_false]]
    is_true = res[0,0] > 0.5
    print(f"'{elem}' is '{pred}'? {is_true} (Scores: True={res[0,0]:.2f}, False={res[1,0]:.2f})")

def check_binary(pred, subj, obj):
    res = model.binaryOp(pred, subj, obj)
    is_true = res[0,0] > 0.5
    print(f"'{subj}' '{pred}' '{obj}'? {is_true} (Scores: True={res[0,0]:.2f}, False={res[1,0]:.2f})")

### Verify Positive Facts
These should return **True**.

In [6]:
# Pick a random positive unary
u_case = [x for x in pos_predicates if x[2] is None][0]
check_unary(u_case[1], u_case[0])

# Pick a random positive binary
b_case = [x for x in pos_predicates if x[2] is not None][0]
check_binary(b_case[1], b_case[0], b_case[2])

'Hazard' is 'is_footballer'? True (Scores: True=1.00, False=0.00)
'hazard' 'won' 'player'? True (Scores: True=1.00, False=0.00)


### Verify Negative Facts
These should return **False**.

In [7]:
# Pick a random negative unary
neg_u_case = [x for x in neg_predicates if x[2] is None][0]
check_unary(neg_u_case[1], neg_u_case[0])

# Pick a random negative binary
neg_b_case = [x for x in neg_predicates if x[2] is not None][0]
check_binary(neg_b_case[1], neg_b_case[0], neg_b_case[2])

'They' is 'is_twin'? False (Scores: True=0.00, False=1.00)
'He' 'tend' 'Wikipedia'? False (Scores: True=0.00, False=1.00)
